# 🧭 Séance 4 — Mini‑projet Titanic : regrouper, croiser, fusionner, dater, puis un tableau enrichi et sa synthèse
_Master IA‑GI — Python pour les sciences de données_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelallioui/python-for-ds-s04/blob/main/mini-projet-04/mini-projet-04.ipynb)

**Comment ouvrir ce notebook :** sur votre ordinateur, ouvrez-le **depuis son dossier** `mini-projet-04/`, sans le déplacer. Les fichiers de départ sont dans `data/`, à côté : `s3_titanic_propre.csv` (celui que votre mini-projet de la séance 3 a produit, en version de référence, pour que tout le monde parte du même point ; **c'est lui qui sert ici**) et `titanic.csv` (le fichier brut, utilisé par la partie 1) ; **vos résultats** sont écrits dans `result/`, à côté aussi. Pandas doit être installé (`pip install pandas` ; déjà présent avec Anaconda).

**Dans Colab**, le notebook arrive **seul** : exécutez la cellule **Étape 0** (plus bas) pour récupérer le dossier du TP, avec ses deux fichiers dans `data/`. Pour **garder votre travail**, faites **Fichier ▸ Enregistrer une copie dans Drive** (ou dans GitHub) : Colab ne sauvegarde pas un notebook ouvert depuis GitHub.

**Objectifs** — à la fin de ce notebook, vous saurez :
- **charger** le tableau propre de la séance 3 et **regrouper** : la survie par classe et par port, avec son **effectif**
- **croiser** deux variables (`pivot_table`, `crosstab`) et **juger** une case par son effectif
- **extraire** le titre caché dans `Name`, puis **affiner** les âges estimés avec `transform` (la médiane **de la classe et du titre**)
- **fusionner** avec une petite table (les escales du Titanic) en **vérifiant la clé**, puis **dater** l'embarquement
- **découper** les âges en tranches, puis **écrire** deux nouveaux fichiers CSV : `s4_titanic_enrichi.csv` et `s4_synthese.csv`, le point de départ de la séance 5

**Durée estimée** : 1 heure environ · **Parties** :
- **M** · Mini-projet : le Titanic enrichi

**Comment travailler seul**
1. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
2. Lisez le *cours en bref*, puis **exécutez le code complet** : c'est lui qui contient les détails des diapos.
3. Faites chaque exercice en remplaçant les `...` par votre code, puis exécutez la cellule : un ✅ ou un ❌ s'affiche.
4. Un ❌ ? Ouvrez l'**indice 1**, puis 2, puis 3.

### 🧰 Pour commencer : importer, puis se contrôler
Exécutez la cellule ci-dessous. Elle importe ce qu'il faut et définit **`verifier`**, la fonction qui affiche ✅ ou ❌ sous chaque exercice. Rien de nouveau dans son écriture : une fonction (`def`), un `try` / `except` et un `lambda`, vus en séance 1.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

print("Pandas", pd.__version__, "· NumPy", np.__version__)

def verifier(ex, description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (même si le test provoque une erreur)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + ex + " · " + description)

### 🪪 Qui êtes-vous ?
Écrivez votre **nom** et votre **prénom** **entre guillemets**, en remplaçant les `...` (par exemple `nom = "Benali"`). Ces deux lignes me permettent de reconnaître votre travail : le contrôle reste ❌ tant qu'elles ne sont pas remplies. **Votre dépôt est public : n'y écrivez jamais de donnée personnelle (CNE, mot de passe, téléphone).**

In [ ]:
nom = ...
prenom = ...

print(nom, prenom)
verifier("identité", "nom et prénom renseignés", lambda: type(nom) is str and type(prenom) is str and len(nom.strip()) >= 2 and len(prenom.strip()) >= 2)

### 📁 Étape 0 · Récupérer le dossier du TP (dans Colab)
Dans **Colab**, le notebook arrive **seul** : les fichiers `titanic.csv` et `s3_titanic_propre.csv` **restent sur GitHub**. Exécutez la cellule ci-dessous : elle **télécharge le dossier du TP** dans la session Colab, puis s'y place. Les fichiers se trouvent alors dans `python-for-ds-s04/mini-projet-04/data/` (panneau **📁** à gauche, bouton **↻** pour actualiser).

**Sur votre ordinateur**, cette cellule ne fait rien : ouvrez simplement le notebook depuis son dossier.

Les fichiers d'une session Colab sont **temporaires** : ils disparaissent à la fermeture. Pour garder un fichier, par exemple `result/s4_synthese.csv` : clic droit dessus ▸ **Télécharger**.

In [ ]:
import os
import subprocess

try:
    import google.colab                 # ce module n'existe que dans Colab
    dans_colab = True
except ImportError:
    dans_colab = False

if dans_colab and not os.path.exists("data/titanic.csv"):
    subprocess.run(["git", "clone", "-q", "https://github.com/yelallioui/python-for-ds-s04"])
    os.chdir("python-for-ds-s04/mini-projet-04")

print("dossier de travail :", os.getcwd())
print("data/titanic.csv trouvé :", os.path.exists("data/titanic.csv"))

**Le fil de data science continue.** À la séance 3, vous avez produit `s3_titanic_propre.csv` : un tableau **propre**, avec ses trous traités **par décision**. Il vous restait quatre questions hors de portée : la survie **par groupe**, l'âge médian **par classe et par titre**, l'**escale** de chaque passager, et les **tranches d'âge**. **Pandas les traite aujourd'hui.**

**La question qui traverse les six séances : qui a survécu au naufrage du Titanic, et pourquoi ?** Vous **regroupez**, **croisez**, **fusionnez** et **datez**, puis vous produisez deux fichiers CSV : `s4_titanic_enrichi.csv` (le tableau propre, **plus cinq colonnes**) et `s4_synthese.csv` (un **tableau de synthèse**, prêt à être mis en graphique en séance 5). Chaque étape se **contrôle toute seule** : un ✅ ou un ❌ vous dit si c'est juste.

**Le fil du mini‑projet**
- **Regrouper et croiser** : **M1** charger le tableau propre · **M2** la survie par classe et par port · **M3** croiser, et lire l'effectif
- **Enrichir** : **M4** extraire le titre, l'âge médian par classe et par titre · **M5** affiner les âges estimés (`transform`)
- **Combiner** : **M6** fusionner avec les escales · **M7** dater l'embarquement
- **Découper** : **M8** les tranches d'âge
- **Résultat** : **M9** `result/s4_titanic_enrichi.csv` et `result/s4_synthese.csv`
- **Fichier de départ** : `data/s3_titanic_propre.csv`, à côté de ce notebook : **c'est à vous de le charger** (M1). _Un seul geste nouveau : `str.extract` (avant M4) ; tout le reste vient de la partie 1 et du cours._

**🧭 Étape M1 — Charger le tableau propre**

**Comment charger `data/s3_titanic_propre.csv`, le tableau propre de la séance 3, dans `df`**, avec le numéro de passager, `PassengerId`, en **index** ? Au final : **891 lignes** et **18 colonnes** (`PassengerId` est l'index) ; les âges n'ont plus de trou (**177 sont des estimations**, marquées `Age_estime`) et `Cabin` a gardé ses **687** trous : c'est ce que vous aviez décidé.

<details>
<summary>💡 Indice 1 · la piste</summary>

Une seule fonction : `pd.read_csv(chemin, index_col=...)`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Le chemin est `"data/s3_titanic_propre.csv"` ; la colonne qui sert d'index s'appelle `PassengerId`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`df = pd.read_csv("data/s3_titanic_propre.csv", index_col="PassengerId")`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
# votre code : lisez le fichier de data/ dans df
df = ...

verifier("M1", "df : 891 passagers, 18 colonnes, numéro de passager en index", lambda: isinstance(df, pd.DataFrame) and df.shape == (891, 18) and df.index.name == "PassengerId")
verifier("M1", "les âges n'ont plus de trou, dont 177 estimés ; Cabin a gardé ses 687 trous", lambda: df["Age"].isna().sum() == 0 and df["Age_estime"].sum() == 177 and df["Cabin"].isna().sum() == 687)
verifier("M1", "les colonnes de la séance 3 sont là : FamilySize, A_cabine, Famille_nombreuse", lambda: {"FamilySize", "A_cabine", "Famille_nombreuse"} <= set(df.columns))

_À partir d'ici, la cellule ci‑dessous refait ce que vous venez d'écrire (`df`) : les étapes suivantes repartent d'une base correcte, même si l'étape M1 vous a résisté._

In [ ]:
df = pd.read_csv("data/s3_titanic_propre.csv", index_col="PassengerId")
print(df.shape)
df.info()

**🧭 Étape M2 — Regrouper : la survie par classe et par port**

Avec `groupby` :
1. `par_classe` : le **taux de survie de chaque classe** (une Series indexée par `Pclass`)
2. `par_port` : un DataFrame indexé par `Embarked`, avec **deux colonnes** : le taux de survie (`mean`) et l'**effectif** (`size`), avec `agg`

_Rappel : `Survived` vaut 1 ou 0, sa moyenne est donc une **proportion**._

<details>
<summary>💡 Indice 1 · la piste</summary>

Trois pièces : la clé, la colonne, la statistique. Pour deux statistiques d'un coup : `agg([...])`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

La clé est `Pclass`, puis `Embarked` ; la colonne `Survived` ; les statistiques `"mean"` et `"size"`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`par_classe = df.groupby("Pclass")["Survived"].mean()` et `par_port = df.groupby("Embarked")["Survived"].agg(["mean", "size"])`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
par_classe = ...
par_port = ...

verifier("M2", "par_classe : une Series de trois classes, 63 %, 47 % et 24 %", lambda: isinstance(par_classe, pd.Series) and list(par_classe.index) == [1, 2, 3] and np.allclose(par_classe.to_numpy(), [0.630, 0.473, 0.242], atol=0.001))
verifier("M2", "par_port : trois ports C, Q, S et deux colonnes, mean et size", lambda: isinstance(par_port, pd.DataFrame) and list(par_port.index) == ["C", "Q", "S"] and list(par_port.columns) == ["mean", "size"])
verifier("M2", "effectifs : 168, 77 et 646 passagers", lambda: list(par_port["size"]) == [168, 77, 646])
verifier("M2", "survie : 55 % à Cherbourg, 39 % à Queenstown, 34 % à Southampton", lambda: np.allclose(par_port["mean"].to_numpy(), [0.554, 0.390, 0.339], atol=0.001))

**🧭 Étape M3 — Croiser, et lire l'effectif**

1. `taux` : le **taux de survie** par classe (lignes) et par port (colonnes), avec `pivot_table`
2. `effectifs` : le **nombre de passagers** par classe et par port, avec `crosstab`
3. `conclusion` : **une phrase** (texte, 30 caractères au moins) qui dit **quelle case du tableau `taux` est la moins fiable** et **pourquoi**

<details>
<summary>💡 Indice 1 · la piste</summary>

Deux tableaux de même forme : trois classes en lignes, trois ports en colonnes. Le premier calcule une **moyenne**, le second **compte**.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.pivot_table(df, values="Survived", index="Pclass", columns="Embarked", aggfunc="mean")` ; `pd.crosstab(df["Pclass"], df["Embarked"])`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

La case la moins fiable : la première classe, port Q. Son taux est de 50 %, mais il repose sur **2 passagers** seulement.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
taux = ...
effectifs = ...
conclusion = ...

verifier("M3", "taux : 3 classes sur 3 ports (C, Q, S)", lambda: isinstance(taux, pd.DataFrame) and taux.shape == (3, 3) and list(taux.columns) == ["C", "Q", "S"])
verifier("M3", "taux : 1re classe à Cherbourg 69 %, 3e classe à Southampton 19 %", lambda: abs(taux.loc[1, "C"] - 0.694) < 0.001 and abs(taux.loc[3, "S"] - 0.190) < 0.001)
verifier("M3", "effectifs : 3 × 3, 891 passagers en tout", lambda: isinstance(effectifs, pd.DataFrame) and effectifs.shape == (3, 3) and int(effectifs.to_numpy().sum()) == 891)
verifier("M3", "effectifs : 2 passagers de 1re classe au port Q", lambda: effectifs.loc[1, "Q"] == 2)
verifier("M3", "conclusion rédigée (au moins 30 caractères)", lambda: isinstance(conclusion, str) and len(conclusion.strip()) >= 30)

---
### Un geste nouveau : `str.extract`
La colonne `Name` cache le **titre** de chaque passager : `Braund, Mr. Owen Harris` → `Mr`. `df["Name"].str.extract(motif, expand=False)` lit un **motif** dans chaque texte et **garde ce qui est entre parenthèses**. Le motif donné ci‑dessous se lit : *une virgule, des espaces, puis tout ce qui précède le premier point*. **Vous n'avez pas à l'écrire.**

Il y a 17 titres différents. On garde les quatre courants (`Mr`, `Mrs`, `Miss`, `Master`) et on range les 13 autres sous `"Autre"` avec `where` : `serie.where(condition, autre_valeur)` garde la valeur quand la condition est vraie, et met `autre_valeur` sinon.

**🧭 Étape M4 — Le titre, puis l'âge médian par classe et par titre**

1. `titre` : une Series de 891 titres : `Mr`, `Mrs`, `Miss`, `Master`, ou `"Autre"` pour tous les autres
2. `age_titre` : l'**âge médian des âges réels** par **classe** (lignes) et par **titre** (colonnes) : un tableau de 3 lignes et 5 colonnes

_Les âges **estimés** en séance 3 valent tous 28 (la médiane globale) : ils fausseraient la médiane. On ne garde donc que les âges **réels** : `ages_reels` vaut `NaN` là où l'âge a été estimé, et `median` ignore les `NaN`._

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour 2, regroupez `ages_reels` par **deux clés** : une colonne de `df` et la Series `titre`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ages_reels.groupby([df["Pclass"], titre]).median()` donne un index à **deux niveaux** ; `unstack()` met le titre en colonnes.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`titre = df["Name"].str.extract(motif, expand=False)` puis `titre = titre.where(titre.isin([...]), "Autre")` ; `age_titre = ages_reels.groupby([df["Pclass"], titre]).median().unstack()`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
motif = r",\s*([^.]+)\."                                  # une virgule, des espaces, puis tout ce qui précède le premier point
titre = ...                                              # extraire, puis ranger les titres rares sous "Autre"
ages_reels = df["Age"].where(~df["Age_estime"])          # l'âge réel ; NaN là où l'âge a été estimé en séance 3
age_titre = ...

verifier("M4", "titre : 891 valeurs, 5 titres possibles", lambda: isinstance(titre, pd.Series) and len(titre) == 891 and set(titre.unique()) == {"Mr", "Mrs", "Miss", "Master", "Autre"})
verifier("M4", "517 Mr, 182 Miss, 125 Mrs, 40 Master, 27 Autre", lambda: titre.value_counts().to_dict() == {"Mr": 517, "Miss": 182, "Mrs": 125, "Master": 40, "Autre": 27})
verifier("M4", "age_titre : 3 classes, 5 titres", lambda: isinstance(age_titre, pd.DataFrame) and age_titre.shape == (3, 5) and list(age_titre.index) == [1, 2, 3])
verifier("M4", "âge médian d'un Mr : 40 ans en 1re classe, 26 ans en 3e", lambda: age_titre.loc[1, "Mr"] == 40.0 and age_titre.loc[3, "Mr"] == 26.0)
verifier("M4", "âge médian d'un Master : 1 an en 2e classe, 4 ans en 3e", lambda: age_titre.loc[2, "Master"] == 1.0 and age_titre.loc[3, "Master"] == 4.0)

_La cellule ci‑dessous refait `titre` et `ages_reels` : les étapes suivantes repartent d'une base correcte._

In [ ]:
motif = r",\s*([^.]+)\."
titre = df["Name"].str.extract(motif, expand=False)
titre = titre.where(titre.isin(["Mr", "Mrs", "Miss", "Master"]), "Autre")
ages_reels = df["Age"].where(~df["Age_estime"])
print(titre.value_counts().to_dict())

**🧭 Étape M5 — Affiner les âges estimés avec transform**

`age_affine` : une Series de 891 âges : l'**âge réel** pour les 714 passagers dont l'âge est connu ; pour les **177 autres**, la **médiane des âges réels de leur classe et de leur titre**, avec `transform`.

_Rappel : `transform("median")` rend **une valeur par ligne** (la médiane de son groupe), contrairement à `agg` qui en rend une par groupe._

<details>
<summary>💡 Indice 1 · la piste</summary>

Deux temps : calculer, **pour chaque passager**, la médiane de son groupe (classe et titre) ; puis remplir les âges manquants de `ages_reels` avec cette médiane.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`mediane_groupe = ages_reels.groupby([df["Pclass"], titre]).transform("median")` ; `fillna` remplit les trous avec une Series, face à chaque étiquette.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`age_affine = ages_reels.fillna(mediane_groupe)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
mediane_groupe = ...
age_affine = ...

verifier("M5", "age_affine : 891 âges, aucun trou", lambda: isinstance(age_affine, pd.Series) and len(age_affine) == 891 and age_affine.isna().sum() == 0)
verifier("M5", "les 714 âges réels sont inchangés", lambda: (age_affine[~df["Age_estime"]] == df.loc[~df["Age_estime"], "Age"]).all())
verifier("M5", "le passager 6 (Mr, 3e classe) reçoit 26 ans, pas 28", lambda: age_affine.loc[6] == 26.0)
verifier("M5", "les 177 âges estimés ont une moyenne de 26,91 ans", lambda: abs(age_affine[df["Age_estime"]].mean() - 26.91) < 0.01)

_La cellule ci‑dessous refait `age_affine` : les étapes suivantes repartent d'une base correcte._

In [ ]:
age_affine = ages_reels.fillna(ages_reels.groupby([df["Pclass"], titre]).transform("median"))
print(age_affine.isna().sum(), "trou ;", round(age_affine[df["Age_estime"]].mean(), 2), "ans en moyenne pour les âges estimés")

---
### Les escales du Titanic
Dans quelle ville et à quelle date chaque passager a‑t‑il embarqué ? Le tableau ne contient que la **lettre** du port (`Embarked`). La petite table ci‑dessous **traduit** chaque lettre : trois lignes, une par escale (le Titanic a quitté **Southampton** le mercredi 10 avril 1912, fait escale à **Cherbourg** le soir même, puis à **Queenstown** le lendemain). Exécutez la cellule.

In [ ]:
escales = pd.DataFrame({
    "Embarked": ["S", "C", "Q"],
    "Ville": ["Southampton", "Cherbourg", "Queenstown"],
    "Date": ["1912-04-10", "1912-04-10", "1912-04-11"],
})
print(escales)

**🧭 Étape M6 — Fusionner avec les escales**

`fusion` : le tableau `df` **joint** à `escales`, pour que chaque passager reçoive sa `Ville` et sa `Date` d'embarquement : une jointure **left** sur la clé `Embarked`, en **vérifiant** qu'une lettre de port correspond à **une seule** ligne d'`escales` (`validate`). `PassengerId` doit rester l'**index** de `fusion`.

_Piège : `merge` **perd l'index** de la table de gauche. On le garde en passant par `reset_index()`, puis on le remet avec `set_index("PassengerId")`._

<details>
<summary>💡 Indice 1 · la piste</summary>

Une grande table (891 lignes) et une petite table de traduction (3 lignes) : c'est un `merge`, avec la clé `Embarked`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.merge(df.reset_index(), escales, on="Embarked", how="left", validate="many_to_one")` : *plusieurs* passagers pour *une* escale. Ensuite, `.set_index("PassengerId")` remet l'index.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`fusion = pd.merge(df.reset_index(), escales, on="Embarked", how="left", validate="many_to_one").set_index("PassengerId")`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
fusion = ...

verifier("M6", "fusion : 891 passagers, les colonnes Ville et Date sont là", lambda: isinstance(fusion, pd.DataFrame) and len(fusion) == 891 and {"Ville", "Date"} <= set(fusion.columns))
verifier("M6", "le numéro de passager est resté l'index, dans le même ordre que df", lambda: fusion.index.name == "PassengerId" and list(fusion.index) == list(df.index))
verifier("M6", "aucune ville manquante : 646 Southampton, 168 Cherbourg, 77 Queenstown", lambda: fusion["Ville"].isna().sum() == 0 and fusion["Ville"].value_counts().to_dict() == {"Southampton": 646, "Cherbourg": 168, "Queenstown": 77})
verifier("M6", "le passager 1 a embarqué à Southampton, le passager 2 à Cherbourg", lambda: fusion.loc[1, "Ville"] == "Southampton" and fusion.loc[2, "Ville"] == "Cherbourg")

_La cellule ci‑dessous refait `fusion` : l'étape suivante repart d'une base correcte, même si l'étape M6 vous a résisté._

In [ ]:
fusion = pd.merge(df.reset_index(), escales, on="Embarked", how="left", validate="many_to_one").set_index("PassengerId")
print(fusion[["Ville", "Date"]].head(3))

**🧭 Étape M7 — Dater l'embarquement**

Dans `fusion` :
1. la colonne `Date_embarquement` : `Date` convertie en **vraies dates**, avec `pd.to_datetime` et un `format`
2. `par_jour` : le **nombre de passagers** embarqués **chaque jour** (une Series indexée par date)
3. `jour_depart` : le **nom du jour** (en anglais) du **premier** embarquement, avec `day_name()`

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour 1 : `pd.to_datetime(colonne, format=...)` ; le format d'une date comme `1912-04-10` est `"%Y-%m-%d"`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Pour 2 : un `groupby` sur la nouvelle colonne, puis `size()`. Pour 3 : la plus petite date (`min()`), puis `day_name()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`fusion["Date_embarquement"] = pd.to_datetime(fusion["Date"], format="%Y-%m-%d")` ; `par_jour = fusion.groupby("Date_embarquement").size()` ; `jour_depart = fusion["Date_embarquement"].min().day_name()`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
fusion["Date_embarquement"] = ...
par_jour = ...
jour_depart = ...

verifier("M7", "Date_embarquement est une colonne de vraies dates", lambda: pd.api.types.is_datetime64_any_dtype(fusion["Date_embarquement"]))
verifier("M7", "par_jour : 814 passagers le 10 avril, 77 le 11 avril", lambda: list(par_jour.to_numpy()) == [814, 77] and len(par_jour) == 2)
verifier("M7", "le premier embarquement a eu lieu un mercredi", lambda: jour_depart == "Wednesday")

_La cellule ci‑dessous refait `fusion` avec ses dates : les étapes suivantes repartent d'une base correcte._

In [ ]:
fusion = pd.merge(df.reset_index(), escales, on="Embarked", how="left", validate="many_to_one").set_index("PassengerId")
fusion["Date_embarquement"] = pd.to_datetime(fusion["Date"], format="%Y-%m-%d")
print(fusion[["Ville", "Date_embarquement"]].head(3))

**🧭 Étape M8 — Les tranches d'âge**

Avec les âges **affinés** (`age_affine`) :
1. `tranche` : une Series de 891 tranches : `Enfant` (de 0 à 12 ans), `Adolescent` (12 à 18), `Adulte` (18 à 60) et `Senior` (60 à 80), avec `pd.cut`
2. `par_tranche` : un DataFrame indexé par tranche, avec le taux de survie (`mean`) et l'**effectif** (`size`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Cinq bornes, quatre noms : `pd.cut(valeurs, bornes, labels=noms)`. Pour regrouper selon une Series qui n'est pas une colonne, on la passe à `groupby`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.groupby(tranche, observed=True)["Survived"].agg(["mean", "size"])` : `observed=True` ne garde que les tranches présentes.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`tranche = pd.cut(age_affine, bornes, labels=noms)` puis `par_tranche = df.groupby(tranche, observed=True)["Survived"].agg(["mean", "size"])`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
bornes = [0, 12, 18, 60, 80]
noms = ["Enfant", "Adolescent", "Adulte", "Senior"]
tranche = ...
par_tranche = ...

verifier("M8", "tranche : 891 valeurs, aucun trou, quatre tranches", lambda: len(tranche) == 891 and tranche.isna().sum() == 0 and set(tranche.astype(str).unique()) == set(noms))
verifier("M8", "par_tranche : quatre lignes, colonnes mean et size", lambda: isinstance(par_tranche, pd.DataFrame) and list(par_tranche.columns) == ["mean", "size"] and list(par_tranche.index) == noms)
verifier("M8", "effectifs : 73 enfants, 103 adolescents, 693 adultes, 22 seniors", lambda: list(par_tranche["size"]) == [73, 103, 693, 22])
verifier("M8", "survie : 58 % des enfants, 23 % des seniors", lambda: abs(par_tranche.loc["Enfant", "mean"] - 0.575) < 0.001 and abs(par_tranche.loc["Senior", "mean"] - 0.227) < 0.001)

**🧭 Étape M9 — Le résultat de la séance : un tableau enrichi et sa synthèse**

Écrivez **`s4_titanic_enrichi.csv`** et **`s4_synthese.csv`** dans le dossier `result/` :
1. `enrichi` : une **copie de `df`** à laquelle vous ajoutez **cinq colonnes**, chacune **face à son numéro de passager** : `Titre` (`titre`), `Age_affine` (`age_affine`), `Ville` et `Date_embarquement` (prises dans `fusion`), et `Tranche_age` (`tranche`) : **891 lignes, 23 colonnes**
2. `synthese` : un tableau **long** de **15 lignes** et quatre colonnes, `Critere`, `Modalite`, `Passagers` et `Survie` : le nombre de passagers et le taux de survie pour chaque modalité de **quatre critères** : la classe (`Pclass`), la ville (`Ville`), la tranche d'âge (`Tranche_age`) et le titre (`Titre`). Une boucle sur les quatre critères, puis `pd.concat`
3. Écrivez `enrichi` dans `chemin_enrichi` (**avec** son index, le numéro de passager) et `synthese` dans `chemin_synthese` (**sans** index), avec `to_csv`

La séance 5 (Matplotlib et Seaborn) reprendra `s4_synthese.csv` pour en faire des graphiques.

<details>
<summary>💡 Indice 1 · la piste</summary>

Pour `enrichi` : `enrichi = df.copy()`, puis `enrichi["Titre"] = titre` : l'**alignement par étiquette** range chaque valeur face à son passager. Pour `synthese` : une liste `morceaux`, un petit `groupby` par critère.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Un morceau : `enrichi.groupby(colonne, observed=True)["Survived"].agg(passagers="size", survie="mean")`, puis `.reset_index()` ; on renomme les trois colonnes (`Modalite`, `Passagers`, `Survie`) et `insert(0, "Critere", nom_du_critere)` ajoute la première colonne.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`for critere, colonne in [("Classe", "Pclass"), ("Ville", "Ville"), ("Tranche d'âge", "Tranche_age"), ("Titre", "Titre")]:` puis, dans la boucle, le morceau, et enfin `synthese = pd.concat(morceaux, ignore_index=True)` (arrondissez `Survie` à 4 décimales avec `round(4)`).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
if Path("data").exists():                                                  # le notebook est bien ouvert depuis son dossier
    Path("result").mkdir(exist_ok=True)                                      # le dossier result, créé s'il manque
chemin_enrichi = Path("result") / "s4_titanic_enrichi.csv"
chemin_synthese = Path("result") / "s4_synthese.csv"

enrichi = ...
synthese = ...
# votre code : écrivez enrichi dans chemin_enrichi et synthese dans chemin_synthese avec to_csv

verifier("M9", "enrichi : 891 lignes, 23 colonnes, les cinq nouvelles colonnes en fin de tableau", lambda: enrichi.shape == (891, 23) and list(enrichi.columns)[-5:] == ["Titre", "Age_affine", "Ville", "Date_embarquement", "Tranche_age"])
verifier("M9", "s4_titanic_enrichi.csv : 891 lignes, 23 colonnes, le numéro de passager en première colonne", lambda: pd.read_csv(chemin_enrichi, index_col="PassengerId").shape == (891, 23))
verifier("M9", "s4_titanic_enrichi.csv : aucun trou dans Titre, Age_affine, Ville, Date_embarquement et Tranche_age", lambda: pd.read_csv(chemin_enrichi)[["Titre", "Age_affine", "Ville", "Date_embarquement", "Tranche_age"]].isna().sum().sum() == 0)
verifier("M9", "synthese : 15 lignes (3 classes, 3 villes, 4 tranches, 5 titres) et les quatre colonnes demandées", lambda: synthese.shape == (15, 4) and list(synthese.columns) == ["Critere", "Modalite", "Passagers", "Survie"] and synthese["Critere"].value_counts().to_dict() == {"Titre": 5, "Classe": 3, "Ville": 3, "Tranche d'âge": 4})
verifier("M9", "synthese : chaque critère compte les 891 passagers, la 1re classe survit à 63 %", lambda: (synthese.groupby("Critere")["Passagers"].sum() == 891).all() and abs(synthese[(synthese["Critere"] == "Classe") & (synthese["Modalite"] == 1)]["Survie"].iloc[0] - 0.6296) < 0.0001)
verifier("M9", "s4_synthese.csv : 15 lignes et les quatre colonnes", lambda: pd.read_csv(chemin_synthese).shape == (15, 4) and list(pd.read_csv(chemin_synthese).columns) == ["Critere", "Modalite", "Passagers", "Survie"])

**🎉 Votre tableau enrichi et sa synthèse sont prêts.** Vous avez `s4_titanic_enrichi.csv` : le tableau propre de la séance 3, **plus cinq colonnes** (le titre, l'âge affiné, la ville et la date d'embarquement, la tranche d'âge) ; et `s4_synthese.csv` : **15 lignes** qui disent, pour chaque classe, ville, tranche d'âge et titre, **combien de passagers** et **quel taux de survie**. Ils sont dans le dossier `result/` de votre mini‑projet.

**Une question de réflexion, sans code :** dans `s4_synthese.csv`, **quelles lignes faut‑il lire avec prudence**, et pourquoi ? _(Par exemple : les 27 passagers « Autre » ou les 22 seniors : un taux calculé sur peu de passagers bouge beaucoup au moindre cas.)_

---
## 🚧 Ce que vous ne savez pas encore faire
Vos tableaux sont prêts, mais ils sont encore des **nombres** : on y lit mal les écarts. Deux questions restent hors de portée :

1. **Comment montrer la survie par classe et par ville** pour qu'on **voie** l'écart au premier coup d'œil, sans lire quinze lignes ? Il faut un **graphique** : le sujet de la **séance 5**.
2. **L'écart entre 63 % et 24 % est‑il réel, ou dû au hasard ?** Seul un **test** peut répondre : le sujet de la **séance 6**.

**À la séance 5**, je vous remettrai le dossier `seance-05`, dont le mini‑projet contient **ce fichier** : `s4_synthese.csv` (et `s4_titanic_enrichi.csv`). **Matplotlib et Seaborn** y transformeront vos tableaux en graphiques lisibles.